In [ ]:
1. Install Required Packages

pip install djangorestframework PyJWT

In [ ]:
2. Simple Custom model

# accounts/models.py
from django.db import models
from django.contrib.auth.hashers import make_password, check_password

class User(models.Model):
    username = models.CharField(max_length=100, unique=True)
    email = models.EmailField(unique=True)
    password = models.CharField(max_length=255)

    def set_password(self, raw_password):
        self.password = make_password(raw_password)

    def check_password(self, raw_password):
        return check_password(raw_password, self.password)

    def __str__(self):
        return self.username


In [ ]:
3. JWT Utility Functions

# accounts/utils.py
import jwt
import datetime
from django.conf import settings

SECRET_KEY = settings.SECRET_KEY

def generate_jwt(user_id):
    payload = {
        'user_id': user_id,
        'exp': datetime.datetime.utcnow() + datetime.timedelta(hours=1),
        'iat': datetime.datetime.utcnow()
    }
    token = jwt.encode(payload, SECRET_KEY, algorithm='HS256')
    return token

def decode_jwt(token):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
        return payload
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None


In [ ]:
4. Serializer

# accounts/serializers.py
from rest_framework import serializers
from .models import User

class UserSerializer(serializers.ModelSerializer):
    class Meta:
        model = User
        fields = ['id', 'username', 'email', 'password']
        extra_kwargs = {'password': {'write_only': True}}

    def create(self, validated_data):
        user = User(
            username=validated_data['username'],
            email=validated_data['email']
        )
        user.set_password(validated_data['password'])
        user.save()
        return user


In [ ]:
5. Authentication Class

# accounts/authentication.py
from rest_framework.authentication import BaseAuthentication
from rest_framework.exceptions import AuthenticationFailed
from .models import User
from .utils import decode_jwt

class JWTAuthentication(BaseAuthentication):
    def authenticate(self, request):
        auth_header = request.headers.get('Authorization')
        if not auth_header:
            return None
        try:
            prefix, token = auth_header.split(' ')
            if prefix.lower() != 'bearer':
                raise AuthenticationFailed('Invalid token prefix')
        except ValueError:
            raise AuthenticationFailed('Invalid token format')

        payload = decode_jwt(token)
        if not payload:
            raise AuthenticationFailed('Invalid or expired token')

        try:
            user = User.objects.get(id=payload['user_id'])
        except User.DoesNotExist:
            raise AuthenticationFailed('User not found')

        return (user, None)


In [ ]:
6. Views (Register, Login, Protected)


class RegisterView(APIView):
    def post(self, request):
        serializer = UserSerializer(data=request.data)
        if serializer.is_valid():
            serializer.save()
            return Response({"message": "User registered successfully"}, status=status.HTTP_201_CREATED)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

class LoginView(APIView):
    def post(self, request):
        username = request.data.get('username')
        password = request.data.get('password')

        try:
            user = User.objects.get(username=username)
        except User.DoesNotExist:
            return Response({"error": "Invalid username or password"}, status=status.HTTP_401_UNAUTHORIZED)

        if not user.check_password(password):
            return Response({"error": "Invalid username or password"}, status=status.HTTP_401_UNAUTHORIZED)

        token = generate_jwt(user.id)
        return Response({"token": token})

class ProtectedView(APIView):
    authentication_classes = [JWTAuthentication]
    permission_classes = [IsAuthenticated]

    def get(self, request):
        return Response({"message": f"Hello, {request.user.username}! You are authenticated."})


In [ ]:
from django.urls import path
from .views import RegisterView, LoginView, ProtectedView

urlpatterns = [
    path('register/', RegisterView.as_view()),
    path('login/', LoginView.as_view()),
    path('protected/', ProtectedView.as_view()),
]

In [ ]:
from django.urls import path, include

urlpatterns = [
    path('api/', include('accounts.urls')),
]

In [ ]:
8. Test the Flow
Register

POST /api/register/
{
    "username": "john",
    "email": "john@example.com",
    "password": "123456"
}

login

POST /api/login/
{
    "username": "john",
    "password": "123456"
}

In [ ]:
pip install djangorestframework djangorestframework-simplejwt

INSTALLED_APPS = [
    # ...
    'rest_framework',
]

from django.contrib.auth import authenticate
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from rest_framework_simplejwt.tokens import RefreshToken

class CustomLoginView(APIView):
    def post(self, request):
        username = request.data.get('username')
        password = request.data.get('password')
        user = authenticate(username=username, password=password)
        if user is not None:
            refresh = RefreshToken.for_user(user)
            return Response({
                'refresh': str(refresh),
                'access': str(refresh.access_token),
                'message': 'Login successful'
            }, status=status.HTTP_200_OK)
        else:
            return Response({'error': 'Invalid Credentials'}, status=status.HTTP_401_UNAUTHORIZED)


In [ ]:
# myapp/middleware.py
import time
from django.utils.deprecation import MiddlewareMixin

class RequestTimingMiddleware(MiddlewareMixin):
    def process_request(self, request):
        request.start_time = time.time()

    def process_response(self, request, response):
        total_time = time.time() - request.start_time
        print(f"[{request.method}] {request.path} took {total_time:.2f}s")
        return response


In [ ]:
process_request
Runs before the view is called.
Stores the start time in request.start_time.
process_response
Runs after the view returns a response.
Calculates and logs the total time taken.
Must return the response.

In [ ]:
import requests
import datetime

STRIPE_SECRET_KEY = "sk_test_your_key_here"  # Replace with your real key
BASE_URL = "https://api.stripe.com/v1/customers"

# Define date range and convert to UNIX timestamps
from_date = "2024-01-01"
to_date = "2024-12-31"
from_ts = int(datetime.datetime.strptime(from_date, "%Y-%m-%d").timestamp())
to_ts = int(datetime.datetime.strptime(to_date, "%Y-%m-%d").timestamp())

headers = {
    "Authorization": f"Bearer {STRIPE_SECRET_KEY}"
}

def fetch_customers(params, collected=None):
    if collected is None:
        collected = []

    response = requests.get(BASE_URL, headers=headers, params=params)
    data = response.json()

    if response.status_code != 200:
        print("Error:", data)
        return collected

    customers = data.get("data", [])
    collected.extend(customers)

    if data.get("has_more") and customers:
        # Recurse with updated `starting_after`
        params["starting_after"] = customers[-1]["id"]
        return fetch_customers(params, collected)
    else:
        return collected

# Initial parameters
initial_params = {
    "limit": 100,
    "created[gte]": from_ts,
    "created[lte]": to_ts
}

# Get all customers
all_customers = fetch_customers(initial_params)

# Output
for customer in all_customers:
    print(f"{customer['id']} - {customer.get('email')} - Created at: {customer['created']}")


In [ ]:
import requests
base_url = ""
params = {
    "limit": 100,
    "created[gte]": from_ts,
    "created[lte]": to_ts
}

headers = {
    "Authorization": f"Bearer {STRIPE_SECRET_KEY}"
}

get_data = requests.get(base_url,headers=headers,params=params)

data = get_data.json()
if data.status_code != 200:
    pass
else:
    customers = data.get("data", [])
    collected.extend(customers)
    